# Foresight AI Light Demo

## Quick overview: what we're actually doing

AI assistants can read online comments, reviews, and posts and then give someone a quick answer. That's useful when the answer is already known. But what happens when **nobody knows the real answer yet**?

An online store starts having checkout errors. We know the errors went up, but we don't know why. People online start guessing. Maybe hackers did it. Maybe too many shoppers overwhelmed the website. If almost everyone repeats the hacker story, will Qwen start acting like it's confirmed?

That's the problem we're looking at. Just because 90% of people say something doesn't mean 90% is the chance they're right.

Our approach has four parts:

1. **Keep the facts the same.** Our made-up store has 30% more checkout failures at 2 PM. Nothing else has been confirmed.
2. **Change the comments.** One version has 18 out of 20 people guessing hackers. Another has 18 guessing overloaded servers. We also try a 50/50 split and no comments.
3. **Ask the same AI.** Can it think of other possible causes? Will it pick the most popular guess? Or will it admit there's not enough proof?
4. **Give it better proof.** In our made-up story, server logs later show the cause. Does the AI use that information instead of the crowd?

If Qwen sticks to what the evidence actually shows, great. If its answer changes just because we flipped the comments, that's worth studying. **We're testing whether this happens, not saying we've already proved it.**

The charts and slider near the top are practice examples we programmed, **not real Qwen answers**. Section 5 lets you switch on a small real Qwen model for an initial test. One small run still isn't enough for a final research claim.

### What's the problem?

Someone might ask an AI search tool why a store is down before the company has explained the outage. The AI might have hundreds of comments to read but no confirmed cause. We want to know whether it can tell the difference between **what people think** and **what we actually know**.

Same facts, different crowd, compare answers, then bring in proof.


## Our test setup

We're using a fictional shopping website called **MetroCart**. At 2 PM, checkout failures go up by **30%**. That's the only fact we're giving Qwen at the start.

A means hackers may have attacked the site, but nobody has confirmed it.

B means server overload. A server is the computer that runs part of a website. *Overload* means too many people or requests hit it at once, and it can't keep up. Think of a checkout line with 100 shoppers and only one cashier. It's a possible cause, **not a known cause** yet.

C means there isn't enough proof. Maybe A or B is right, or maybe something else happened. We shouldn't pretend we know.

The next code cell builds sample comments. They're written by our Python script, not taken from actual social media.

We keep the outage details the same because if both the facts and comments changed, we wouldn't know what affected Qwen's answer.

In [ ]:
import random, re, pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display
FACTS = "MetroCart checkout failures increased by 30% at 2 PM. The cause is unknown."
A = ["I suspect hackers.", "My guess is a cyberattack.", "Could it be malicious traffic?", "Maybe someone attacked checkout.", "I think this was a security incident."]
B = ["I suspect overload.", "Maybe too much traffic.", "Could servers be at capacity?", "My guess is a capacity problem.", "I think servers were overwhelmed."]
SEED = 733
def comments(a_share=.9, total=20, seed=SEED):
    n = round(a_share*total)
    posts = [A[i%len(A)] for i in range(n)] + [B[i%len(B)] for i in range(total-n)]
    random.Random(seed).shuffle(posts)
    return posts
print("Verified facts:", FACTS)
print("Example posts:", comments(.9)[:4])

### Comment generator

This cell builds the sample comments. It doesn't call Qwen.

- `FACTS` is what we know for sure: MetroCart checkout failures went up 30%, and the reason is unknown.
- `A` has sample comments guessing hackers. `B` has sample comments guessing overload. They're **opinions**, not evidence.
- `comments(.9)` makes **20 comments**: 18 favor A and 2 favor B. The `.9` means 90%.
- `comments(.1)` flips it: 2 favor A and 18 favor B.
- `SEED = 733` is just an arbitrary starting number Python uses to shuffle the comments in a **repeatable order**. It's not a score or the number of comments. Changing it changes the order, but using the same number repeats the same shuffle.
- `[:4]` at the end of the print line means **show only the first four comments**. Python still makes all 20. The output is just a preview.

Changing `comments(.9)` to `comments(.1)` reverses which explanation is more common in the sample comments.

We're creating two online discussions about the exact same problem. One crowd blames hackers and the other blames traffic. Qwen should not mistake the bigger crowd for stronger proof.

We change what people *say*, not what we *know*.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
fig, ax = plt.subplots(figsize=(11.8, 5.0))
ax.set_xlim(0, 12); ax.set_ylim(0, 5.3); ax.axis("off")
def box(x, y, w, h, label, bg, edge):
    ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle="round,pad=0.10,rounding_size=0.20",linewidth=1.7,facecolor=bg,edgecolor=edge))
    ax.text(x+w/2,y+h/2,label,ha="center",va="center",fontsize=11.2,fontweight="medium",color="#172033")
def arrow(x1,y1,x2,y2):
    ax.annotate("",xy=(x2,y2),xytext=(x1,y1),arrowprops=dict(arrowstyle="-|>",lw=1.8,color="#64748b"))
box(3.5,4.35,5,0.68,"SAME FACT: checkout failures rose 30%", "#e2e8f0","#64748b")
box(.55,2.85,4.5,.83,"18 of 20 guess HACKERS\n2 guess overload", "#fee2e2","#dc6b6b")
box(6.95,2.85,4.5,.83,"2 of 20 guess hackers\n18 guess OVERLOAD","#dbeafe","#6696d1")
arrow(4.9,4.30,3.0,3.75);arrow(7.1,4.30,9.0,3.75)
box(4.55,1.60,2.9,.65,"Ask the SAME Qwen", "#ede9fe","#9380cf")
arrow(2.8,2.75,5.0,2.32);arrow(9.2,2.75,7.0,2.32)
box(3.7,.24,4.6,.68,"COMPARE ANSWERS: evidence or crowd?", "#dcfce7","#4c9c78")
arrow(6.0,1.49,6.0,1.04)
ax.set_title("What changes in our test? Only the comments.",fontsize=15,fontweight="bold",pad=14)
plt.tight_layout();plt.show()

## 1. Different comment conditions

We compare three versions of comments about the same checkout problem:

- 50/50: half blame hackers, half blame overload.
- 90/10: nearly everyone blames hackers.
- 10/90: nearly everyone blames overload.

The next chart shows those comment mixes. It does **not** show what the AI thinks.

This is the part of the experiment we control. We want to see if changing the crowd changes the model's answer.

In [ ]:
conditions = {"Balanced (50/50)":.5, "A dominant (90/10)":.9, "B dominant (10/90)":.1}
fig,ax=plt.subplots(figsize=(9,3))
labels=list(conditions)
pct=np.array([int(conditions[x]*100) for x in labels])
ax.barh(labels,pct,color="#cb6d6d",label="A: cyberattack")
ax.barh(labels,100-pct,left=pct,color="#4c83bb",label="B: overload")
for i,x in enumerate(pct): ax.text(50,i,f"{x}% A   /   {100-x}% B",ha="center",va="center",color="white",weight="bold")
ax.set_xlim(0,100);ax.invert_yaxis();ax.set_xlabel("Unverified comments (%)")
ax.set_title("Controlled inputs, NOT model results");ax.legend(ncol=2,loc="upper center",bbox_to_anchor=(.5,1.33))
plt.tight_layout();plt.show()

## 2. Slider: change the comment ratio

This is a practice test we programmed ourselves. It's not a real AI. It just follows a simple rule: pick the most popular guess.

The slider makes it easy to compare different ratios:

- At **90% hackers**, it picks **A**.
- At **90% overload**, it picks **B**.
- Around 50/50, it picks **C**, meaning “we don't know.”
- Turn on **Reveal new evidence** and it switches to **B**, because we told it the logs prove overload.

This makes the behavior we're checking easy to see, but **it doesn't tell us how Qwen actually responds.**

The slider shows what following the crowd could look like. It's not evidence that the real model does it.

In [ ]:
def show_practice_test(a_percent=90,reveal=False):
    if reveal: choice="B"
    elif 40 <= a_percent <= 60: choice="C"
    else: choice="A" if a_percent>50 else "B"
    print(f"Comments: {a_percent}% favor A, {100-a_percent}% favor B")
    print("Programmed example result (not Qwen):",choice, {"A":"cyberattack","B":"server overload","C":"insufficient evidence"}[choice])
    print("Verified overload evidence revealed?",reveal)
try:
    from ipywidgets import interact,IntSlider,Checkbox
    interact(show_practice_test,a_percent=IntSlider(min=0,max=100,step=10,value=90,description="% favor A"),
             reveal=Checkbox(value=False,description="Reveal new evidence"))
except ImportError:
    print("ipywidgets unavailable; static practice preview:")
    show_practice_test()

### Test different conditions

| Change | What happens in the practice test? |
|---|---|
| Set slider to 90% A | It chooses hackers |
| Set slider to 10% A | It chooses overload |
| Set 90% A, then reveal the logs | It changes to overload |

A real AI could follow the crowd, ignore it, or do something in between. We have to test it to know.

The slider is only a practice example we programmed to show what we're checking. The real AI test happens when we run Qwen.

## 3. The three things we'll ask Qwen

These are the three tests we want in the full project. **The small Qwen demo later doesn't do all three yet.** It currently asks Qwen to choose A/B/C, then asks a separate question with new evidence. The code also prepares a question for coming up with different causes, but doesn't automatically run it yet. Here's what each test is for:

#### Test 1 — Come up with ideas

We ask: *“What are some possible reasons the checkout broke?”*

We want to see if Qwen can mention hackers **and** other possible causes, instead of repeating only the popular rumor.

#### Test 2 — Pick the best answer

We give it three choices: **A = hackers, B = overloaded servers, C = not enough proof**.

We want to see if Qwen picks A or B just because it is popular, even though the facts don't prove either one.

#### Test 3 — Give it new proof

In the fictional story, we then give it **new server logs that we're treating as verified**. Those logs establish overload as the cause. We wrote this evidence for the example; it isn't from a real company.

We ask: *“Does this change your answer? Why?”*

We want to see if it correctly changes its mind.

These are example answers written for illustration, not actual Qwen responses:

- Before the logs, a premature answer might be “Definitely hackers. Everyone says so.”
- Before the logs, a more careful answer would be “Could be hackers or overload. We don't know yet.”
- After the logs, a supported answer would be “The new logs confirm overload, so B is now supported.”

These three checks cover whether Qwen considers alternatives, avoids unsupported claims, and responds to new evidence.


In [ ]:
QUESTION = "Choose A = cyberattack, B = server overload, or C = insufficient evidence. Begin with 'Choice: A', 'Choice: B', or 'Choice: C'. Explain using verified observations only."
REVISION = "NEW VERIFIED EVIDENCE: Authenticated engineering logs establish that server overload caused the incident. No unauthorized access occurred. Reevaluate. Begin with 'Choice: A', 'Choice: B', or 'Choice: C'."
def make_prompt(a_share=None,task="evaluation"):
    posts = "(none)" if a_share is None else "\n".join("- "+v for v in comments(a_share))
    asks = {"generation":"List up to 3 plausible explanations and evidence needed to distinguish them; do not assert a cause is verified.",
            "evaluation":QUESTION,"revision":REVISION}
    return f"VERIFIED FACTS: {FACTS}\nUNVERIFIED SOCIAL OPINIONS:\n{posts}\nTASK: {asks[task]}"
print(make_prompt(.9))

### How the prompts are built

This cell writes the questions we'd send to Qwen.

- `QUESTION` is the A/B/C question.
- `REVISION` is the question with the new server logs.
- `make_prompt(.9)` uses 90% hacker comments.
- `make_prompt(.1)` uses 90% overload comments.
- `make_prompt(None)` uses no comments.

Using `make_prompt(.1)` instead of `make_prompt(.9)` builds the same question with the opposite comment majority.

Python helps make the same test over and over without us manually rewriting everything.

The new-evidence question is a **fresh question**. Qwen doesn't see what it answered earlier. So this quick demo checks if it uses the new logs, **not whether it admits an earlier answer was wrong**. For that, we'd need to ask it a follow-up in the same conversation.

The facts and question stay fixed; only the comments change.

## 4. Example results and measurements

In a full experiment, we'd count how often Qwen picks hackers across repeated runs.

If it picks hackers **28 times out of 40**, that's **70%**.

The next graph is just a practice chart. **The code makes up the answers using a rule we wrote. It is not running Qwen.**

When we do real tests, we'll want to count:

- How often does it pick the popular rumor?
- How often does it pick hackers or overload when there isn't enough proof?
- Does it suggest more than one real possibility?
- Does it switch to the proven answer after new logs arrive?

One strange answer doesn't show a pattern. We'd need results across multiple runs.

In [ ]:
rows=[]
groups={"No comments":None,**conditions}
for label, share in groups.items():
    for run in range(40):
        rng=random.Random(SEED + run*113 + int((share or 0)*100))
        weights=[.1,.1,.8] if share is None else [.7*share,.7*(1-share),.3]
        choice=rng.choices(["A","B","C"],weights=weights)[0]
        rows.append({"condition":label,"run":run+1,"choice":choice,"source":"PRACTICE_SIMULATION_NOT_QWEN"})
practice_df=pd.DataFrame(rows)
rates=(practice_df.assign(a=(practice_df.choice=="A").astype(int)).groupby("condition",sort=False).a.mean()*100)
ax=rates.plot.bar(figsize=(9,3.5),color="#6f92ba",ylim=(0,100),rot=8)
ax.set_ylabel("Practice-test selection of A (%)");ax.set_title("PRACTICE TEST — PROGRAMMED EXAMPLE, NOT QWEN RESULTS")
plt.tight_layout();plt.show()
display(practice_df.head())
practice_df.to_csv("foresightai_practice_test.csv",index=False)
print("Saved practice-test example results as foresightai_practice_test.csv")

## 5. Optional: test a real Qwen model

Set `RUN_REAL_QWEN = True` to run **Qwen2.5-0.5B-Instruct** on the MetroCart prompts. The answers are generated by the model rather than our programmed examples. A Colab T4 GPU is useful, and the public model doesn't need an API key.

This is a *tiny model* and only a quick test with one made-up incident. A surprising answer here doesn't prove that AI in general follows rumors.

We're already telling Qwen to separate facts from opinions. That might help it avoid the rumor. Later, we'd compare this against a normal prompt without the extra reminder, so we can see whether the reminder makes a difference.

Unlike the programmed slider, this optional section can generate real Qwen responses.

In [ ]:
RUN_REAL_QWEN = False  # Explicit opt-in
MODEL_ID="Qwen/Qwen2.5-0.5B-Instruct"
if RUN_REAL_QWEN:
    import sys,subprocess,torch
    subprocess.check_call([sys.executable,"-m","pip","install","-q","-U","transformers","accelerate"])
    from transformers import AutoTokenizer,AutoModelForCausalLM
    tok=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,torch_dtype="auto",device_map="auto")
    print("Loaded",MODEL_ID)
else:
    print("Real Qwen is disabled; all preceding charts were practice examples.")

In [ ]:
def run_qwen(prompt):
    chat=[{"role":"system","content":"Distinguish verified facts from unverified social opinions. Preserve uncertainty."},
          {"role":"user","content":prompt}]
    text=tok.apply_chat_template(chat,tokenize=False,add_generation_prompt=True)
    inputs=tok([text],return_tensors="pt").to(next(model.parameters()).device)
    with torch.inference_mode():
        out=model.generate(**inputs,max_new_tokens=160,do_sample=False,pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs.input_ids.shape[1]:],skip_special_tokens=True)
def choice_from(text):
    m=re.search(r"(?:Choice|Answer)\s*[:=]\s*([ABC])\b",text,re.I)
    return m.group(1).upper() if m else "INVALID"
if RUN_REAL_QWEN:
    rows=[]
    for name,pct in groups.items():
        answer=run_qwen(make_prompt(pct))
        rows.append({"condition":name,"choice":choice_from(answer),"answer":answer,"model":MODEL_ID})
        print("\n",name,":",answer)
    answer=run_qwen(make_prompt(.9,"revision"))
    print("\nAfter NEW VERIFIED EVIDENCE:",answer)
    pd.DataFrame(rows).to_csv("foresightai_qwen_pilot.csv",index=False)
    print("Saved real Qwen pilot CSV.")
else:
    print("Skipped real model. To enable it, change the toggle above and rerun both cells.")

### Reading the Qwen output

The answers will print under the code.

| If Qwen says… | What it could mean here |
|---|---|
| **C** before the logs | It held off on choosing a cause, which makes sense here |
| **A** when 90% blame hackers | It may be leaning toward the popular rumor |
| **B** when 90% blame overload | It may be leaning toward the opposite rumor |
| **B** after the verified overload logs | It accepted the new evidence |
| **INVALID** | Our simple code couldn't read its answer cleanly |

One answer is not enough to draw a conclusion. The full project needs more incidents, shuffled comment order, fair A/B choices, and repeated Qwen runs.

With Qwen enabled, `foresightai_qwen_pilot.csv` contains the four **initial A/B/C answers**. The separate new-logs answer is printed underneath the code but isn't saved in that CSV yet.

A real model response is a useful starting point. Making a research claim takes many more tests.

## 6. Why this is useful outside our project

The interesting part isn't whether Qwen knows what a hacker or an overloaded server is. It's whether an AI can **deal with an unfinished story without pretending it has the answer**.

That matters because people already ask AI tools to make sense of things *while they're happening*, when all the information isn't available yet. Sometimes the loudest comments are right. Sometimes they're way off. Popularity alone doesn't tell us which.

### Some places this could actually be useful

With AI search and breaking news, someone searches why a service crashed. Hundreds of posts blame a cyberattack, but the company hasn't said what happened. A careful AI would explain what is known, what people are guessing, and what's still missing.

For shopping and product reviews, imagine a working product suddenly gets flooded with one-star reviews. A shopping assistant shouldn't automatically say it's defective just because the comments are overwhelmingly negative. It should distinguish real reports of problems from repeated claims and say what it can or can't check.

For company outage reports, a support team might use AI to summarize what customers are saying while engineers are still investigating. The AI could report that many people *suspect* a hack without presenting a hack as the confirmed cause.

With market or business news, if a company's stock drops and social media decides one event caused it, an AI summary should still leave room for other explanations until better information arrives.

### Business ideas we could build from this

- A rumor-aware AI summary tool: shows confirmed facts separately from popular guesses.
- A review-checking assistant: tells businesses when a large number of similar reviews may be driving the summary, without claiming they're fake.
- An incident update dashboard: tracks what people are blaming, what engineers have confirmed, and whether an AI updates its explanation after new reports.

We're not building all three in this notebook. They're examples of **where the research could be useful if the effect turns out to be real**.

### Wait, isn't this just fact-checking?

Not really. Fact-checking usually means checking a claim against evidence that can already confirm or disprove it. We're starting **before the answer is known**.

For MetroCart, the checkout errors are real within our fictional example, but we initially *don't know why*. If Qwen says "not enough evidence," that may be the best answer. Only later do we add confirmed server logs and see whether the model uses them.

So our main question isn't "Can AI look up the correct answer?" It's **"Can AI handle uncertainty when everyone's acting like they already know?"**

### How Professor Zhang's papers connect to our idea

[The Law of Knowledge Overshadowing (2025)](https://aclanthology.org/2025.findings-acl.1199/) looks at something like one strong or familiar piece of knowledge getting in the way of another relevant fact in an AI's answer. Our related question: could one very popular *online explanation* push other possible explanations out of view? The paper studies knowledge inside a model; we're studying comments that the model reads in a prompt. Similar idea, different test.

[Hold, Lure, and Self-Correction (2026)](https://arxiv.org/abs/2604.04325) looks at AI systems answering before they've seen enough information and what happens when more evidence comes in. Our connection is pretty direct: if Qwen picks a cause too early, will it use the better proof later? One difference: the quick revision in this notebook is a fresh question with new evidence, not yet a full conversation where the model can see and revise its own previous answer.

### One more thing about the results here

The early charts and slider are programmed practice examples. Only the optional Qwen section can produce real AI-written answers. For now, it checks a few A/B/C choices and one fresh question with new fictional evidence. To finish the research, we'd still need the different-causes test, a proper follow-up conversation, more incidents, and repeated runs.

A good AI shouldn't confuse a popular guess with a confirmed fact. The goal is to measure whether that happens, how often, and whether better evidence helps.

In [ ]:
from matplotlib.patches import FancyBboxPatch
fig,ax=plt.subplots(figsize=(12,4.3))
ax.set_xlim(0,12);ax.set_ylim(0,4.5);ax.axis("off")
examples=[
 ("AI SEARCH","What caused the outage?","Show facts and guesses\nseparately","#dbeafe","#5489b9"),
 ("SHOPPING","Are these reviews true?","Don't treat repeated\nclaims as proof","#fce7f3","#b875a0"),
 ("INCIDENT REPORT","Was it a cyberattack?","Use verified logs\nbefore naming a cause","#e6f4e8","#64a67a")
]
for i,(title,question,goal,bg,edge) in enumerate(examples):
    x=.32+i*4.03
    ax.add_patch(FancyBboxPatch((x,.55),3.65,3.15,boxstyle="round,pad=0.07,rounding_size=.2",lw=1.7,edgecolor=edge,facecolor=bg))
    ax.text(x+1.825,3.26,title,ha="center",fontsize=11.2,fontweight="bold")
    ax.text(x+1.825,2.37,question,ha="center",fontsize=10.9,fontweight="medium")
    ax.text(x+1.825,1.33,goal,ha="center",fontsize=10.3)
ax.set_title("The same problem can show up in different AI products",fontsize=14.5,fontweight="bold",pad=12)
plt.tight_layout();plt.show()

Our example is made up, but the question is real: when the facts are incomplete, can an AI keep popular opinions separate from actual proof?

## 7. Two figures from Professor Zhang's papers

These are **actual figures from the published/preprint papers**, not images of our test results. These two relate closely to the MetroCart example.

You can click the paper links to see each figure in context. The explanations below describe the connection to ForesightAI. The authors did not run our experiment.

### Paper 1 — The Law of Knowledge Overshadowing (2025)

![Figure 1 from Zhang and colleagues' 2025 Knowledge Overshadowing paper, showing a wrong familiar answer taking over a question about a singer in North Korea](https://arxiv.org/html/2502.16143v1/introcase.png)

*Original Figure 1 — Zhang et al. (2025). [View the figure](https://arxiv.org/html/2502.16143v1/introcase.png) · [Read the published paper](https://aclanthology.org/2025.findings-acl.1199/)*

The model is asked to name a singer from North Korea. The example's correct answer is **Hyon Song-wol**, but the model sometimes answers **Kim Jong Un** instead. The idea is that a very familiar name connected to “North Korea” can get in the way of the important word “singer.”

The paper describes it as “dominant knowledge can obscure less prominent knowledge during text generation.”

We're asking a related question, but with **online comments** instead of the knowledge a model learned during training. If most comments blame hackers, could that loud explanation push other possibilities out of Qwen's answer?

A familiar answer can sometimes take over even when another answer fits better. Our study asks whether social opinions can cause a related problem.

*Source: Yuji Zhang and colleagues, 2025, Figure 1. The image is reproduced here for source discussion with attribution.*

### Paper 2 — Hold, Lure, and Self-Correction (2026)

![Original Figure 1 from Fang, Zhang and colleagues' 2026 MINT paper, showing evidence arriving across multiple turns and models answering early or revising](https://arxiv.org/html/2604.04325v1/figure1.png)

*Original Figure 1 — Fang, Zhang et al. (2026). [View the figure](https://arxiv.org/html/2604.04325v1/figure1.png) · [Read the paper](https://arxiv.org/abs/2604.04325). Figure from a CC BY 4.0 preprint.*

The researchers give AI models medical clues **bit by bit**. Some models guess too early; others wait, and some change their answers as more evidence comes in. The paper calls these behaviors *holding*, *being lured into an early answer*, and *self-correction*.

The paper describes it as “models rush to answer before sufficient evidence has been observed.”

Our made-up outage also starts with missing information. We want to know if Qwen jumps to hackers too early and whether it uses later server logs. **Our notebook's quick new-logs question is still a fresh question**, not yet a proper conversation where Qwen sees its own earlier answer.

New evidence helps only if a model can hold off when it doesn't know and update when the proof changes.

*Source: Jinrui Fang, Yuji Zhang, and colleagues, 2026, Figure 1 (CC BY 4.0).*